In [3]:
pip install requests beautifulsoup4 pandas nltk matplotlib seaborn wordcloud

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\55119\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import nltk
import os

# Define o caminho para a nossa pasta local
# O '.' representa o diretório atual do projeto
local_nltk_path = os.path.join('nltk_data')

# Verifica se a pasta existe, se não, cria
if not os.path.exists(local_nltk_path):
    os.makedirs(local_nltk_path)

# Diz ao NLTK para usar APENAS este caminho
# (ou para procurar aqui primeiro)
nltk.data.path.insert(0, local_nltk_path)

# Baixa os pacotes para o nosso diretório customizado
print(f"Baixando pacotes para: {os.path.abspath(local_nltk_path)}")
nltk.download('punkt', download_dir=local_nltk_path)
nltk.download('stopwords', download_dir=local_nltk_path)

print("\nDownload e configuração de caminho concluídos!")

Baixando pacotes para: c:\Users\55119\OneDrive\Área de Trabalho\Projetos em andamento\portfolio\notebooks\nltk_data

Download e configuração de caminho concluídos!


[nltk_data] Downloading package punkt to nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
import pandas as pd

# Carrega os dados coletados na Fase 1
df = pd.read_csv('../data/ml_reviews_raw.csv')

# Exibe as 5 primeiras linhas para verificar
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   title   0 non-null      float64
 1   text    9 non-null      object 
 2   date    9 non-null      object 
dtypes: float64(1), object(2)
memory usage: 348.0+ bytes
None
   title                                               text          date
0    NaN  Produto √© bonito e rapido tanto para ligar ou...  16 mar. 2025
1    NaN                               Estou adorando üòä.  14 mar. 2025
2    NaN  Aparelho √© sensacional!!!.\nA mem√≥ria √© coi...  17 mar. 2025
3    NaN  Aparelho √≥timo, bate de frente com aparelhos ...  09 mai. 2025
4    NaN  Poco x7 pro aparelho espetacular, comprem sem ...  06 mai. 2025


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer # <-- Importamos o novo tokenizador

# --- CARREGAMENTO APENAS DAS STOPWORDS ---
# O único recurso externo que ainda precisamos são as stopwords.
# Mantenha o caminho que já estava funcionando para elas.
try:
    path_to_stopwords = r'C:\Users\55119\AppData\Roaming\nltk_data\corpora\stopwords\portuguese'
    with open(path_to_stopwords, 'r', encoding='utf-8') as f:
        stop_words = set(f.read().splitlines())
    print("Stopwords carregadas com sucesso!")
except Exception as e:
    print(f"Erro ao carregar stopwords: {e}")
    stop_words = set() # Define um conjunto vazio em caso de falha

# --- FUNÇÃO DE LIMPEZA FINAL ---
def clean_text(text):
    """
    Função de limpeza final que usa RegexpTokenizer para evitar o erro do pickle.
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text, re.I|re.A)
    
    # Cria um tokenizador que considera apenas sequências de letras como palavras
    # Isso efetivamente remove números e o que sobrou de pontuação.
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(text)
    
    # Filtra as stopwords
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    clean_text = ' '.join(filtered_tokens)
    
    return clean_text

Stopwords carregadas com sucesso!


In [7]:
import pandas as pd
# Verificando se a coluna 'text' existe e não tem valores nulos para evitar erros
# A função .dropna() remove as linhas onde a coluna 'text' é vazia
df.dropna(subset=['text'], inplace=True)

# Aplica a função de limpeza na coluna 'text' e cria a coluna 'text_clean'
df['text_clean'] = df['text'].apply(clean_text)

# Exibe o antes e o depois para comparar
print("\nComparando texto original com o texto limpo:")
print(df[['text', 'text_clean']].head())


Comparando texto original com o texto limpo:
                                                text  \
0  Produto √© bonito e rapido tanto para ligar ou...   
1                               Estou adorando üòä.   
2  Aparelho √© sensacional!!!.\nA mem√≥ria √© coi...   
3  Aparelho √≥timo, bate de frente com aparelhos ...   
4  Poco x7 pro aparelho espetacular, comprem sem ...   

                                          text_clean  
0  produto bonito rapido tanto ligar reiniciar bo...  
1                                           adorando  
2  aparelho sensacional memria coisa doido super ...  
3  aparelho timo bate frente aparelhos dobro preo...  
4  poco x pro aparelho espetacular comprem medo r...  


In [8]:
# Salva o DataFrame processado em um novo arquivo para uso futuro
# Este arquivo agora contém a coluna 'text_clean'
output_path_processed = '../data/ml_reviews_processed.csv'
df.to_csv(output_path_processed, index=False, encoding='utf-8-sig')

print(f"DataFrame processado e salvo com sucesso em: {output_path_processed}")

DataFrame processado e salvo com sucesso em: ../data/ml_reviews_processed.csv
